# SENTIRA — Section 3B Reliability Audit

*(Filename kept as `Audio_Suggestion_testing.ipynb` for continuity with prior versions — consider renaming to `section3B_reliability_audit.ipynb` if you reorganize the repo later.)*

This notebook contains **every experiment behind the Section 3B audit and several other reviewer comments** — it is the single most important notebook for responding to the review, and was previously the hardest to locate because most of its cells had no headers. It is now organized into six parts:

- **Part A** — setup (load the trained Audio fusion model + data)
- **Part B** — the audio/semantic audit: branch ablation, standalone branches, the TF-IDF content-only control, and the Wav2Vec2-vs-TF-IDF agreement test (M8)
- **Part C** — independent fusion cross-check, bootstrap confidence intervals (source of Table 3.6), and the majority-vote baseline comparison
- **Part D** — data-leakage / split-integrity sanity check
- **Part E** — seed-robustness check
- **Part F** — EEG attention weights and the full EEG stream ablation (source of Section 3A's raw-EEG chance-level figure)

**Requires (upload when prompted):** `subject_split.json`, `scalers.pkl`, `best_audio_model.pt`, `audio_features.h5`, `uc1_predictions.npz`, `uc3_predictions_v2.npz`, `uc2_predictions_shared_subjects.npz`, `eeg_fusion_wrapper_v2.pkl`, `eeg_features_test_only.h5`.

## PART A — Setup: Mount Drive + Locate Files
(Upload: `subject_split.json`, `scalers.pkl`, `best_audio_model.pt`, `audio_features.h5`)

In [2]:
from google.colab import files
uploaded = files.upload()

Saving audio_fusion_wrapper.pkl to audio_fusion_wrapper.pkl
Saving best_audio_model.pt to best_audio_model.pt
Saving best_emotion_model.pt to best_emotion_model.pt
Saving checkpoint.db to checkpoint.db
Saving scalers.pkl to scalers.pkl
Saving subject_split.json to subject_split.json
Saving training_history.json to training_history.json


In [3]:
from google.colab import files
uploaded = files.upload()

Saving audio_features.h5 to audio_features.h5


In [4]:
import os

for f in sorted(os.listdir("/content")):
    path = os.path.join("/content", f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"{f:35s} {size_mb:.2f} MB")

audio_features.h5                   73.27 MB
audio_fusion_wrapper.pkl            17.48 MB
best_audio_model.pt                 17.43 MB
best_emotion_model.pt               1.82 MB
checkpoint.db                       1.17 MB
scalers.pkl                         0.05 MB
subject_split.json                  0.00 MB
training_history.json               0.00 MB


## PART A.2 — Imports + Model Architecture (must match the trained checkpoint exactly)

In [5]:
import torch, torch.nn as nn
import numpy as np, json, pickle, h5py
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import TensorDataset, DataLoader

class BranchEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim * 2), nn.BatchNorm1d(hidden_dim * 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim), nn.BatchNorm1d(hidden_dim),
            nn.GELU(), nn.Dropout(dropout * 0.75),
        )
    def forward(self, x): return self.net(x)

class AttentionFusionModel(nn.Module):
    def __init__(self, trad_dim=558, w2v_dim=1024, sem_dim=768,
                 hidden_dim=512, num_classes=5, dropout=0.40):
        super().__init__()
        self.trad_enc = BranchEncoder(trad_dim, hidden_dim, dropout)
        self.w2v_enc  = BranchEncoder(w2v_dim,  hidden_dim, dropout)
        self.sem_enc  = BranchEncoder(sem_dim,  hidden_dim, dropout)
        self.attn_gate = nn.Sequential(
            nn.Linear(hidden_dim * 3, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 3), nn.Softmax(dim=1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 256), nn.BatchNorm1d(256), nn.GELU(),
            nn.Dropout(dropout * 0.75),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.GELU(),
            nn.Dropout(dropout * 0.50),
            nn.Linear(128, num_classes),
        )
    def forward(self, trad, w2v, sem, return_attn=False):
        h_t = self.trad_enc(trad); h_w = self.w2v_enc(w2v); h_s = self.sem_enc(sem)
        cat = torch.cat([h_t, h_w, h_s], dim=1)
        weights = self.attn_gate(cat)
        fused = h_t*weights[:,0:1] + h_w*weights[:,1:2] + h_s*weights[:,2:3]
        logits = self.classifier(fused)
        if return_attn: return logits, weights
        return logits

## PART A.3 — Load Subject Split, Scalers, and Trained Model Checkpoint

In [6]:
with open("/content/subject_split.json") as f:
    split_info = json.load(f)
print("Test subjects:", split_info["test"])

with open("/content/scalers.pkl", "rb") as f:
    scalers = pickle.load(f)

ckpt = torch.load("/content/best_audio_model.pt", map_location="cpu")
state_dict = ckpt.get("model_state_dict", ckpt) if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
print("Sample state_dict keys:", list(state_dict.keys())[:5])
print("Total keys:", len(state_dict))

Test subjects: ['subject15', 'subject16', 'subject18', 'Subject1', 'Subject7', 'subject41']
Sample state_dict keys: ['trad_enc.net.0.weight', 'trad_enc.net.0.bias', 'trad_enc.net.1.weight', 'trad_enc.net.1.bias', 'trad_enc.net.1.running_mean']
Total keys: 62


## PART B.1 — Branch-Ablation on the Trained Fusion Model  
**Resolves Comment 10/24 (semantic-only after scrubbing).** Zeroes out one or two branches at inference time (no retraining) to see how much each branch contributes to the already-trained model's decision. `Only Semantic` here is the source of the paper's reported **97.67%** figure.

In [7]:
### PART B.2 — Load Full Train+Val+Test Features (no subject filter — needed for standalone branch training below)

✅ Loaded 600 test samples across 6 subjects
Feature dims -> trad:558  w2v:1024  sem:768

--- ABLATION RESULTS (zeroed-branch diagnostic on trained fusion model) ---
Full model:         97.5
Semantic zeroed:    30.666666666666664
Trad zeroed:        98.5
W2V zeroed:         97.5
Only Semantic:      97.66666666666667
Only Trad+W2V:      30.666666666666664


In [8]:
### PART B.3 — Standalone Single-Branch Classifiers  
Trains a **fresh** classifier per branch (Traditional / Wav2Vec2 / Semantic) from scratch, independent of the fusion model — gives each branch's own best-case ceiling.

Train: 3000 Val: 600 Test: 600


In [9]:
# ── Fresh single-branch classifier (NOT competing with other branches) ──
class SingleBranchModel(nn.Module):
    def __init__(self, in_dim, hidden_dim=512, num_classes=5, dropout=0.4):
        super().__init__()
        self.enc = BranchEncoder(in_dim, hidden_dim, dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.BatchNorm1d(128), nn.GELU(),
            nn.Dropout(dropout*0.5), nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.head(self.enc(x))

def train_single_branch(X_all, y_all, scaler, epochs=60, lr=3e-4, name=""):
    Xtr = scaler.fit_transform(X_all[train_mask])
    Xva = scaler.transform(X_all[val_mask])
    Xte = scaler.transform(X_all[test_mask])
    ytr, yva, yte = all_y[train_mask], all_y[val_mask], all_y[test_mask]

    model = SingleBranchModel(in_dim=Xtr.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    crit = nn.CrossEntropyLoss()

    Xtr_t, ytr_t = torch.tensor(Xtr), torch.tensor(ytr)
    Xva_t, yva_t = torch.tensor(Xva), torch.tensor(yva)
    Xte_t, yte_t = torch.tensor(Xte), torch.tensor(yte)

    best_val_acc, best_state, patience_ctr = 0, None, 0
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), 64):
            idx = perm[i:i+64]
            opt.zero_grad()
            out = model(Xtr_t[idx])
            loss = crit(out, ytr_t[idx])
            loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            val_acc = accuracy_score(yva_t, model(Xva_t).argmax(1))
        sched.step(val_acc)
        if val_acc > best_val_acc:
            best_val_acc, best_state, patience_ctr = val_acc, model.state_dict(), 0
        else:
            patience_ctr += 1
            if patience_ctr > 15: break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_acc = accuracy_score(yte_t, model(Xte_t).argmax(1)) * 100
    print(f"{name:20s} standalone test accuracy: {test_acc:.2f}%")
    return test_acc

from sklearn.preprocessing import StandardScaler

acc_trad = train_single_branch(all_trad, all_y, StandardScaler(), name="Traditional-only")
acc_w2v  = train_single_branch(all_w2v,  all_y, StandardScaler(), name="Wav2Vec2-only")
acc_sem  = train_single_branch(all_sem,  all_y, StandardScaler(), name="Semantic-only")

Traditional-only     standalone test accuracy: 59.33%
Wav2Vec2-only        standalone test accuracy: 95.50%
Semantic-only        standalone test accuracy: 99.67%


In [10]:
### PART B.4 — TF-IDF Content-Only Classifier: Setup Notes

In [11]:
### PART B.5 — Load Transcripts from HDF5 (these are the **scrubbed** transcripts — see note below)

Total transcripts loaded: 4200
Empty/missing transcripts: 0
emotion
A    55.996429
C    48.969048
H    53.947619
N    43.661905
S    50.983333
Name: transcript, dtype: float64

     emotion                                         transcript
3603       A  Seriously, the problem is your complete lack o...
781        C  Absolutely, all king in the forest is one of m...
2165       N  In the tables have evolved over time and accom...
1217       C  The water was crystal-y transparent and blue. ...
1006       A  Hey, are you serious right now? What's your pr...
941        C  Gazing at the stars has a calming effect on me...
2589       H  You are right. Let's not overthink every step....
2801       A  We're not falling for it anymore. You're every...
2871       A  I understand that you have your own ideas, but...
1333       H  It would be so magical. That sounds really ama...


In [12]:
### PART B.6 — TF-IDF-Only Classifier (content, no audio at all)  
**Resolves Comment 32.** Confirms the transcripts loaded above are scrubbed (emotion words replaced with `[EMO]` at extraction time — see the main Audio notebook's `extract_semantic()`), so the **99.81%** result here is the scrubbed-transcript figure, and is the stronger of the two possible readings the reviewer asked about.

Using 4200 samples with non-empty transcripts
TF-IDF-only (content, no audio) accuracy per fold: [0.99777778 0.99666667 0.99875    0.99875    0.99875   ]
Mean: 99.81%  |  Chance level: 20.00%


### PART B.7 — M8: Wav2Vec2-only vs TF-IDF-only Agreement on the Same Test Trials  
**Resolves Comment 34/M8.** Retrains Wav2Vec2-only (saving its test predictions this time), gets TF-IDF's out-of-fold predictions on the same trials, and measures how often the two agree. High agreement (95.50%) supports the audit's conclusion that Wav2Vec2's high accuracy comes from the same lexical/phonetic script-identity shortcut as the semantic branch — not a separate acoustic-emotion mechanism.

In [13]:
# ── M8: Wav2Vec2-only vs TF-IDF-only agreement on the SAME test trials ──
import numpy as np
import torch
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_predict
from sklearn.preprocessing import StandardScaler

# 1) Re-train Wav2Vec2-only, this time SAVING predictions + subjects for the test set
def train_single_branch_with_preds(X_all, y_all, scaler, epochs=60, lr=3e-4, name=""):
    Xtr = scaler.fit_transform(X_all[train_mask])
    Xva = scaler.transform(X_all[val_mask])
    Xte = scaler.transform(X_all[test_mask])
    ytr, yva, yte = all_y[train_mask], all_y[val_mask], all_y[test_mask]

    model = SingleBranchModel(in_dim=Xtr.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    crit = torch.nn.CrossEntropyLoss()

    Xtr_t, ytr_t = torch.tensor(Xtr), torch.tensor(ytr)
    Xva_t, yva_t = torch.tensor(Xva), torch.tensor(yva)
    Xte_t, yte_t = torch.tensor(Xte), torch.tensor(yte)

    best_val_acc, best_state, patience_ctr = 0, None, 0
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xtr_t))
        for i in range(0, len(perm), 64):
            idx = perm[i:i+64]
            opt.zero_grad()
            out = model(Xtr_t[idx])
            loss = crit(out, ytr_t[idx])
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_acc = accuracy_score(yva_t, model(Xva_t).argmax(1))
        sched.step(val_acc)
        if val_acc > best_val_acc:
            best_val_acc, best_state, patience_ctr = val_acc, model.state_dict(), 0
        else:
            patience_ctr += 1
            if patience_ctr > 15: break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_pred = model(Xte_t).argmax(1).numpy()
    test_acc = accuracy_score(yte_t.numpy(), test_pred) * 100
    print(f"{name:20s} standalone test accuracy: {test_acc:.2f}%")
    return test_pred, yte_t.numpy()

w2v_test_pred, w2v_test_true = train_single_branch_with_preds(
    all_w2v, all_y, StandardScaler(), name="Wav2Vec2-only")
w2v_test_subjects = all_subj[test_mask]

# 2) TF-IDF: out-of-fold predictions for every sample, then filter to test subjects only
df_valid = df_t[df_t["transcript"].str.strip() != ""].reset_index(drop=True)
vec = TfidfVectorizer(max_features=2000, ngram_range=(1,2))
X_text = vec.fit_transform(df_valid["transcript"])
y_text = df_valid["label"].values
groups = df_valid["subject"].values

gkf = GroupKFold(n_splits=5)
clf = LogisticRegression(max_iter=1000, C=1.0)
tfidf_oof_pred = cross_val_predict(clf, X_text, y_text, cv=gkf, groups=groups)

test_subject_names = set(split_info["test"])
mask_test_rows = df_valid["subject"].isin(test_subject_names).values

tfidf_test_pred = tfidf_oof_pred[mask_test_rows]
tfidf_test_true = y_text[mask_test_rows]
tfidf_test_subjects = df_valid["subject"].values[mask_test_rows]

print(f"\nWav2Vec2 test rows: {len(w2v_test_pred)}   TF-IDF test rows (filtered): {len(tfidf_test_pred)}")

# 3) Safety checks — must both be True for the agreement number to be valid
print("Order match (same subject sequence):", (w2v_test_subjects == tfidf_test_subjects).all())
print("Label match (same true labels)     :", (w2v_test_true == tfidf_test_true).all())

# 4) Agreement + joint error matrix
agreement = (w2v_test_pred == tfidf_test_pred).mean() * 100
print(f"\nAgreement between Wav2Vec2-only and TF-IDF-only: {agreement:.2f}%")

cm = confusion_matrix(w2v_test_pred, tfidf_test_pred)
print("\nJoint prediction matrix (rows=Wav2Vec2 pred, cols=TF-IDF pred):")
print(cm)

Wav2Vec2-only        standalone test accuracy: 95.33%

Wav2Vec2 test rows: 600   TF-IDF test rows (filtered): 600
Order match (same subject sequence): True
Label match (same true labels)     : True

Agreement between Wav2Vec2-only and TF-IDF-only: 95.50%

Joint prediction matrix (rows=Wav2Vec2 pred, cols=TF-IDF pred):
[[111   0   1   1   0]
 [  3 119   5   5   1]
 [  2   1 114   1   0]
 [  1   1   0 110   0]
 [  2   0   0   3 119]]


In [ ]:
## PART C — Fusion Cross-Check: Setup (upload `uc1_predictions.npz`, `uc3_predictions_v2.npz`, `uc2_predictions_shared_subjects.npz`)

Saving fusion_results_UC4_UC7.json to fusion_results_UC4_UC7.json
Saving fusion_results_both_methods.json to fusion_results_both_methods.json
Saving fusion_confidence_gated_result.json to fusion_confidence_gated_result.json


In [ ]:
### PART C.1 — (Optional) Load Any Previously Saved Fusion-Result JSONs, If Present

=== fusion_results_UC4_UC7.json ===
{
  "UC4_audio_video": {
    "n": 400,
    "accuracy": 0.995,
    "macro_f1": 0.995,
    "weighted_f1": 0.995,
    "kappa": 0.9938
  },
  "UC5_audio_eeg": {
    "n": 400,
    "accuracy": 0.995,
    "macro_f1": 0.995,
    "weighted_f1": 0.995,
    "kappa": 0.9938
  },
  "UC6_video_eeg": {
    "n": 400,
    "accuracy": 0.5325,
    "macro_f1": 0.5107,
    "weighted_f1": 0.5107,
    "kappa": 0.4156
  },
  "UC7_all_three": {
    "n": 400,
    "accuracy": 0.995,
    "macro_f1": 0.995,
    "weighted_f1": 0.995,
    "kappa": 0.9938
  }
}

=== fusion_results_both_methods.json ===
{
  "static_performance_weighted": {
    "UC4_audio_video": {
      "n": 400,
      "accuracy": 0.995,
      "macro_f1": 0.995,
      "weighted_f1": 0.995,
      "kappa": 0.9938
    },
    "UC5_audio_eeg": {
      "n": 400,
      "accuracy": 0.995,
      "macro_f1": 0.995,
      "weighted_f1": 0.995,
      "kappa": 0.9938
    },
    "UC6_video_eeg": {
      "n": 400,
      "accuracy"

In [ ]:
### PART C.2 — Re-upload Prediction Files if Needed

Saving uc2_predictions_shared_subjects.npz to uc2_predictions_shared_subjects.npz
Saving uc3_predictions_v2.npz to uc3_predictions_v2.npz


In [ ]:
### PART C.3 — Independent Re-Verification of UC6 Fusion + McNemar Tests  
Recomputes the Video+EEG fusion (UC6) from the raw saved prediction arrays as an independent cross-check of Fusion.ipynb's result, and re-runs the McNemar significance tests reported in Section 3C.

UC6 accuracy: 53.25
UC6 (Video+EEG) vs UC2 (Video alone): χ²=9.0115  p=2.6829e-03  ✅ SIGNIFICANT
UC6 (Video+EEG) vs UC3 (EEG alone): χ²=17.7632  p=2.5018e-05  ✅ SIGNIFICANT


<bunch containing results, print to see contents>

## PART C.4 — Subject-Level Bootstrap Confidence Intervals (UC2, UC3, UC6)  
**This is the source of Table 3.6's confidence intervals.** 2,000 resamples of the 4 aligned subjects, with replacement.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

def subject_bootstrap_ci(y_true, y_pred, subject_ids, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    unique_subs = np.unique(subject_ids)
    accs = []
    for _ in range(n_boot):
        sampled_subs = rng.choice(unique_subs, size=len(unique_subs), replace=True)
        mask = np.concatenate([np.where(subject_ids == s)[0] for s in sampled_subs])
        accs.append(accuracy_score(y_true[mask], y_pred[mask]))
    lo, hi = np.percentile(accs, [2.5, 97.5])
    return np.mean(accs)*100, lo*100, hi*100

# subj_ids for the aligned video/EEG/UC6 data — extract from common_ids
subj_ids_aligned = np.array([cid[0] for cid in common_ids])  # subject number per sample

acc_uc2, lo2, hi2 = subject_bootstrap_ci(y_true_a, uc2_pred, subj_ids_aligned)
acc_uc3, lo3, hi3 = subject_bootstrap_ci(y_true_a, uc3_pred, subj_ids_aligned)
acc_uc6, lo6, hi6 = subject_bootstrap_ci(y_true_a, uc6_pred, subj_ids_aligned)

print(f"UC2 (Video): {acc_uc2:.2f}%  95% CI [{lo2:.2f}%, {hi2:.2f}%]")
print(f"UC3 (EEG):   {acc_uc3:.2f}%  95% CI [{lo3:.2f}%, {hi3:.2f}%]")
print(f"UC6 (V+E):   {acc_uc6:.2f}%  95% CI [{lo6:.2f}%, {hi6:.2f}%]")

UC2 (Video): 45.96%  95% CI [36.75%, 52.50%]
UC3 (EEG):   41.66%  95% CI [35.00%, 48.50%]
UC6 (V+E):   53.19%  95% CI [35.75%, 62.50%]


In [ ]:
### PART C.5 — Audio's Own Accuracy on the 4 Truly-Shared Subjects (15, 16, 18, 41)

Audio samples on 4 shared subjects: 400
Audio accuracy on this subset: 99.50%


In [ ]:
### PART C.6 — Majority Vote Across All Three Branches + McNemar vs. Audio-Always Baseline  
Source of the paper's **71.50% majority-vote** and **99.50% audio-always baseline** figures (Table 3.5), and the χ²=108.08 McNemar result.

Audio table entries: 400
IDs in common_ids but missing from audio: 0
IDs in audio but not in common_ids:       0
✅ Perfect alignment — safe to proceed.
Aligned audio accuracy check: 99.50%  (should be ~99.50%)

Majority vote (audio+video+eeg): 71.50%
Audio-always naive baseline:      99.50%
Majority-vote vs Audio-always: χ²=108.0789  p=2.5828e-25  ✅ SIGNIFICANT


In [ ]:
## PART D — Data-Leakage / Split-Integrity Check  
Confirms there is no case-sensitivity or other overlap bug between the Audio branch's train/val/test subject sets (a sanity check, independent of any specific reviewer comment, but supports the paper's subject-independence claims).

Train∩Test (case-insensitive): set()
Val∩Test (case-insensitive): set()
Train∩Val (case-insensitive): set()

Raw split sizes: 30 6 6
Test subjects raw: ['subject15', 'subject16', 'subject18', 'Subject1', 'Subject7', 'subject41']


In [ ]:
## PART E — Robustness Check: Wav2Vec2-only Accuracy Across 4 Random Seeds  
Confirms the Wav2Vec2-only standalone result isn't a lucky seed (95.13% ± 0.41% across seeds 13/42/77/101).

seed=13: 95.17%
seed=42: 95.67%
seed=77: 94.50%
seed=101: 95.17%

Wav2Vec2-only across 4 seeds: 95.13% ± 0.41%


In [ ]:
## PART F — EEG Attention Weights + Stream Ablation: Setup (upload `uc3_predictions_v2.npz`, `eeg_fusion_wrapper_v2.pkl`, `eeg_features_test_only.h5`)

Saving uc3_predictions_v2.npz to uc3_predictions_v2 (1).npz


In [ ]:
import os
print(os.getcwd())
for f in os.listdir("."):
    print(f)

/content
.config
subject_split.json
fusion_results_both_methods.json
uc2_predictions_shared_subjects.npz
best_audio_model.pt
best_emotion_model.pt
scalers.pkl
uc3_predictions_v2 (1).npz
audio_features.h5
fusion_confidence_gated_result.json
uc3_predictions_v2.npz
drive
training_history.json
fusion_results_UC4_UC7.json
checkpoint.db
audio_fusion_wrapper.pkl
sample_data


In [ ]:
### PART F.1 — EEG Attention Weights (Fixed-Split Model)  
**Supports Comment 10 (stream collapse).** Reads the already-saved attention weights for the reported EEG model — raw EEGNet stream receives ≈0% weight.

Keys in file: ['y_true', 'y_pred', 'y_probs', 'test_subjects', 'attention_weights', 'sample_keys']

EEGNet (raw): 0.00%
DE features:  25.37%
PSD features: 74.63%

Full model accuracy (from saved preds): 42.42%


In [ ]:
### PART F.2 — Attention Weights, Higher Precision (min/max across samples)

EEGNet (raw): 0.000000  (min=0.000000, max=0.000000)
DE features:  0.253683
PSD features: 0.746317


In [ ]:
### PART F.3 — Install h5py (if not already available)

In [ ]:
### PART F.4 — Re-upload EEG Model Wrapper + Test-Only Features (if needed)

Saving eeg_features_test_only.h5 to eeg_features_test_only.h5


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving eeg_fusion_wrapper_v2.pkl to eeg_fusion_wrapper_v2.pkl


In [ ]:
## PART F.5 — Full EEG Stream Ablation (zeroed-input, no retraining)  
**Resolves Comment 29/M6b.** Loads the trained EEG fusion model and zeroes out each stream in turn. `Raw EEG only: 20.00%` is the exact chance-level figure quoted in Section 3A — this cell is its source. (Note: this prints accuracy only; the per-sample predicted-class distribution the reviewer specifically asked about was not saved — add a `Counter(y_pred)` line here to get it if needed for a follow-up response.)

Using device: cpu
✅ Model + scalers loaded
✅ Test set: 2400 epochs across 6 subjects

--- EEG STREAM ABLATION ---
Full model:     42.42
Raw EEG zeroed: 42.38
DE zeroed:      40.25
PSD zeroed:     21.67
Raw EEG only:   20.00
DE only:        20.50
PSD only:       42.04
